In [6]:
### Download Dogs vs Cats Zip ###
!wget https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

### Download IMDB Dataset ###
!wget http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

### Clean-Up CatsVDogs Images
!wget https://raw.githubusercontent.com/priyammaz/HAL-DL-From-Scratch/main/prep_data.py
!python -m prep_data --catsvdogs

--2025-06-26 02:47:52--  https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
Resolving download.microsoft.com (download.microsoft.com)... 23.1.108.212, 2a02:26f0:6d00:3b6::317f, 2a02:26f0:6d00:39f::317f
Connecting to download.microsoft.com (download.microsoft.com)|23.1.108.212|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 824887076 (787M) [application/octet-stream]
Saving to: ‘kagglecatsanddogs_5340.zip’

kagglecatsanddogs_5 100%[===================>] 786.67M   242MB/s    in 3.3s    

2025-06-26 02:47:55 (238 MB/s) - ‘kagglecatsanddogs_5340.zip’ saved [824887076/824887076]

--2025-06-26 02:47:55--  http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘acl

In [7]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision.datasets import ImageFolder # Stream data from images stored in folders

import os
import numpy as np
from PIL import Image
from collections import Counter

# DataLoaders for Computer Vision

## Dogs Vs Cats Dataset

In our Cats vs Dogs dataset, this is what our file directory looks like:

```
.
└── data/
    └── Petimages/
        ├── Dogs/
        │   ├── xxx.jpg
        │   ├── yyy.jpg
        │   └── ...
        └── Cats/
            ├── xxx.jpg
            ├── yyy.jpg
            └── ...
```

In our folder Petimages, we have two more folders called Dogs and Cats, and each folder contains images of Dogs and Cats respectively. So we have two options:
- Load all our images into Numpy Arrays (and tensors later) up front along with their class label 0 or 1 **### PLEASE DONT DO THIS!!!**
- Store only a list of strings that indicate the path to each Image and then load the image only when accessed


### Components of a Dataset

The dataset class will have three components:
- **init**: Initialize the model class with everything you want to store as class variables
- **len**: We have to tell the Dataset how many samples there are. When we go to grab a sample, it can grab all the samples from index 0 to index len
- **getitem**: This is the block that does most of the heavy lifting. Basically, given some index between 0 and the length of data defined before, it will grab some sample.
```
class CustomDataset(Dataset):
    def __init__(self):
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        pass
```

### Lets be a bit more specific about whats going on!

```
.
└── data/
    └── Petimages/
        ├── Dogs/
        │   ├── 111.jpg
        │   ├── 222.jpg
        └── Cats/
            ├── 333.jpg
            ├── 444.jpg
```

Lets say in our directories we have only 4 images in total (2 Cat and 2 Dog Images). Our **len** would then be 4 as we only have 4 images in total! The **getitem** will then use the indexes [0,1,2,3] to access these samples, so our goal in the getitem is to give the Dataset a way to load the image based on the sample number.

The easiest way to do this is to store a list of filepaths to all of the images, and then when we access the filepath, we can load that image in! So we would first have to build this list of paths and store it to some class variable we can access later!

**Note**: This function should not be in the **getitem**. You want it to happen when the Dataset is being Initialized, otherwise we will build the list every time we grab a sample. The code in **getitem** runs **EVERY TIME WE GRAB A SAMPLE**, but **init** will only run once.

```
path_to_dogs = [data/Petimages/Dogs/111.jpg, data/Petimages/Dogs/222.jpg] # Make a list of the path to all the Dog files
path_to_cats = [data/Petimages/Cats/333.jpg, data/Petimages/Dogs/444.jpg] # Make a list of the path to all the Cat files

training_files = path_to_dogs + path_to_cats = [data/Petimages/Dogs/111.jpg, data/Petimages/Dogs/222.jpg,
                                                data/Petimages/Cats/333.jpg, data/Petimages/Dogs/444.jpg]
                                                
```

Now that we have a list of files, we have to set the **len**, which in this case would just be the length of training_files, or 4.

Lastly, when we start doing a forloop through our Dataset, the **getitem** function is run with an input of **idx**. More specifically, as we loop from 0 to 3 (our total samples of 4 as indicated from before), the idx returned for us to access in **getitem** can be utilized. For example, the first loop will return the index 0, and we can then take that and index our training files.

- In the first iteration at the index 0 we have the filepath *data/Petimages/Dogs/111.jpg*.
- In the second iteration at the index 1 we have the filepath *data/Petimages/Dogs/222.jpg*.
- In the third iteration at the index 2 we have the filepath *data/Petimages/Cats/333.jpg*.
- In the fourth iteration at the index 3 we have the filepath *data/Petimages/Cats/444.jpg*.

Once we do the fourth iteration, we have done the entire length as indicated in **len** and the loop will end. Therefore in **getitem**, if we can index the filepath to each sample in this way, we can then load the image from the file and return the image as an array of numbers. Also, we can easily find the label of the image (Is it a Cat or Dog?) because the filepath includes "Dogs" and "Cats" in the name. Therefore we can also return from **getitem** some interger values like 0, if Dog is in the path, or 1 if Cat is in the path.

#### Arrays vs Tensors
When we load an image we will use the PIL Image module that we imported above. All this module does is take a filepath and loads the image in the PIL format. We can then convert this to a numpy array, but numpy arrays dont work with PyTorch, so we need to convert to a tensor. From PyTorch Torchvision module, we have imported transform which has a ton of cool image transformations we can do (and will look at a bit later). The one we need right now is the **ToTensor()** function that can accept a PIL image (or Numpy Array) and convert to a Tensor that PyTorch models can work with!


#### 8Bit Images
Most images are stored in what is known as an 8bit format. Essentially each pixel in the image can take integer values in the range of [0,255]. Now the problem with this is, Deep Learning tends to prefer numbers scaled between [0,1], so we just need to scale our 8bit images down. One way is to just divide everything by 255, but the **ToTensor()** function will already handle this for us in the PIL -> Tensor transformation.

### Lets Put all the ideas Together!!

In [8]:
class DogsVsCats(Dataset):
    def __init__(self, path_to_folder, transform = None):
        path_to_cats = os.path.join(path_to_folder, 'Cat')
        path_to_dogs = os.path.join(path_to_folder, 'Dog')

        cat_files = os.listdir(path_to_cats)
        dog_files = os.listdir(path_to_dogs)

        path_to_cat_files = [os.path.join(path_to_cats, file) for file in cat_files]
        path_to_dog_files = [os.path.join(path_to_dogs, file) for file in dog_files]

        self.training_files = path_to_cat_files + path_to_dog_files

        self.dog_label = 0
        self.cat_label = 1

        # Use custom transform if provided, otherwise default to ToTensor
        if transform is not None:
            self.transform = transform
        else:
            self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.training_files)

    def __getitem__(self, idx):
        path_to_image = self.training_files[idx]

        if 'Dog' in path_to_image:
            label = self.dog_label
        else:
            label = self.cat_label

        image = Image.open(path_to_image)
        image = self.transform(image)

        return image, label

In [9]:
path_to_folder = '/content/PetImages'

In [10]:
full_dataset = DogsVsCats(path_to_folder)

In [11]:
# Calculate split sizes
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
test_size = total_size - train_size

print(f"Total images: {total_size}")
print(f"Training images: {train_size}")
print(f"Testing images: {test_size}")

Total images: 24998
Training images: 19998
Testing images: 5000


### We have a Dataset class built and now lets try loading it to the DataLoader!

To reiterate, we have built a dataloader that can access a single image, but in Deep Learning, we want to sample minibatches, so we need to grab a **BATCH_SIZE** amount of images. We can do that pretty easily using the DataLoader!

The Dataloader has some more functions we will look at later but the most basic things we need to include are:

```
DataLoader(dataset=DATASET, # We place here the dataset we have defined previously
           batch_size=16,   # How many samples do you want to put together in each batch?
           shuffle=True)    # Do you want to shuffle the data?
```

Lets then go ahead and instantiate the dataloder and try to do a for loop. Again we are expecting 16 images at once!


## Common PyTorch Image Transformations (for torchvision.transforms)

### `ToTensor()`
- Converts a **PIL Image** or **NumPy ndarray** into a **PyTorch Tensor**.
- Scales image pixel values from `[0, 255]` to `[0.0, 1.0]`.

---

### `Resize()`
- Resizes an input image to a given size.
- Useful for datasets with **inconsistent image dimensions**.

```python
transforms.Resize((height, width))
````

---

### `Normalize(mean, std)`

* Normalizes a tensor image with mean and standard deviation.
* Each channel (R, G, B) is normalized using:

  $$
  \text{image} = \frac{\text{image} - \text{mean}}{\text{std}}
  $$
* Helps the model train faster and more stably.

```python
transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
```

---

### `RandomHorizontalFlip(p=0.5)`

* Randomly flips the image **horizontally** with a default probability of **0.5**.
* Useful for **data augmentation** to improve model generalization.

---

### `RandomVerticalFlip(p=0.5)`

* Randomly flips the image **vertically** with a default probability of **0.5**.
* Also used for **data augmentation**, especially in cases where vertical orientation is not important.

---

### `Compose([...])`

* Combines multiple transformations into **a single pipeline**.
* Example:

```python
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
```

* Applies transformations **in the given order**.



In [12]:
# Training transforms (with augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Test transforms (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [13]:
batch_size=64
num_workers=2

In [14]:
# Split the dataset indices
train_indices, test_indices = random_split(
    range(total_size),
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # For reproducibility
)

In [15]:
# Create datasets
train_dataset = DogsVsCats(path_to_folder, transform=train_transform)
test_dataset = DogsVsCats(path_to_folder, transform=test_transform)

# Create subset datasets using the indices
train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)


# Create data loaders
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

In [16]:
for images, labels in train_loader:
    print(images.shape)
    print(labels)
    break

torch.Size([64, 3, 224, 224])
tensor([0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0,
        0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1])


## With ImageFolder

In [17]:
dataset_path =  '/content/PetImages'

# Define transforms for training (with data augmentation)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define transforms for testing (no augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create full dataset to get total length and class info
full_dataset = ImageFolder(dataset_path, transform=transforms.ToTensor())

# Get class information
class_names = full_dataset.classes
class_to_idx = full_dataset.class_to_idx

print(f"Classes found: {class_names}")
print(f"Class to index mapping: {class_to_idx}")

# Calculate split sizes
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
test_size = total_size - train_size

print(f"\nTotal images: {total_size}")
print(f"Training images: {train_size}")
print(f"Testing images: {test_size}")

# Split the dataset indices
train_indices, test_indices = random_split(
    range(total_size),
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # For reproducibility
)

# Create separate datasets with appropriate transforms
train_dataset = ImageFolder(dataset_path, transform=train_transform)
test_dataset = ImageFolder(dataset_path, transform=test_transform)

# Create subset datasets using the indices
train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

# Create data loaders
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True if torch.cuda.is_available() else False
)

Classes found: ['Cat', 'Dog']
Class to index mapping: {'Cat': 0, 'Dog': 1}

Total images: 24998
Training images: 19998
Testing images: 5000


# DataLoaders for NLP

In [18]:
imdb_data_path = '/content/aclImdb/train'

path_to_pos = os.path.join(imdb_data_path, 'pos')
path_to_neg = os.path.join(imdb_data_path, 'neg')

path_to_pos_text = [os.path.join(path_to_pos, file) for file in os.listdir(path_to_pos)]
path_to_neg_text = [os.path.join(path_to_neg, file) for file in os.listdir(path_to_neg)]

In [21]:
training_files = path_to_pos_text + path_to_neg_text

In [25]:
all_text = ''
for file in training_files:
    with open(file, 'r') as f:
        text = f.readlines()
        all_text += text[0]

In [26]:
unique_counts = dict(Counter(all_text))
characters = sorted([key for (key, value) in unique_counts.items() if value > 1500])
characters.append('<UNK>')
characters.append('<PAD>')
char_to_idx = {c:i for i, c in enumerate(characters)}
idx_to_char = {i:c for i, c in enumerate(characters)}

In [27]:
print(char_to_idx)

{' ': 0, '!': 1, '"': 2, '&': 3, "'": 4, '(': 5, ')': 6, '*': 7, ',': 8, '-': 9, '.': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, ';': 23, '<': 24, '>': 25, '?': 26, 'A': 27, 'B': 28, 'C': 29, 'D': 30, 'E': 31, 'F': 32, 'G': 33, 'H': 34, 'I': 35, 'J': 36, 'K': 37, 'L': 38, 'M': 39, 'N': 40, 'O': 41, 'P': 42, 'R': 43, 'S': 44, 'T': 45, 'U': 46, 'V': 47, 'W': 48, 'Y': 49, 'Z': 50, 'a': 51, 'b': 52, 'c': 53, 'd': 54, 'e': 55, 'f': 56, 'g': 57, 'h': 58, 'i': 59, 'j': 60, 'k': 61, 'l': 62, 'm': 63, 'n': 64, 'o': 65, 'p': 66, 'q': 67, 'r': 68, 's': 69, 't': 70, 'u': 71, 'v': 72, 'w': 73, 'x': 74, 'y': 75, 'z': 76, 'é': 77, '<UNK>': 78, '<PAD>': 79}


In [32]:
import torch
from torch.utils.data import Dataset
import os

class IMDBDataset(Dataset):
    def __init__(self, imdb_data_path, char_to_idx, max_length=1000):
        self.char_to_idx = char_to_idx
        self.max_length = max_length

        path_to_pos = os.path.join(imdb_data_path, 'pos')
        path_to_neg = os.path.join(imdb_data_path, 'neg')

        path_to_pos_text = [os.path.join(path_to_pos, file) for file in os.listdir(path_to_pos)]
        path_to_neg_text = [os.path.join(path_to_neg, file) for file in os.listdir(path_to_neg)]

        self.training_files = path_to_pos_text + path_to_neg_text

    def __len__(self):
        return len(self.training_files)

    def __getitem__(self, idx):
        path_to_text = self.training_files[idx]
        with open(path_to_text, 'r', encoding='utf-8') as f:
            text = f.read()[:self.max_length]  # truncate long reviews

        # Tokenize to indices
        tokenized = [self.char_to_idx.get(char, self.char_to_idx["<UNK>"]) for char in text]

        # Pad sequence
        if len(tokenized) < self.max_length:
            tokenized += [self.char_to_idx["<PAD>"]] * (self.max_length - len(tokenized))

        input_tensor = torch.tensor(tokenized, dtype=torch.long)

        label = 0 if 'neg' in path_to_text else 1
        return input_tensor, torch.tensor(label, dtype=torch.long)

In [34]:
import string

# Basic character set (ASCII letters, digits, punctuation)
all_chars = list(string.ascii_letters + string.digits + string.punctuation + ' ')
char_to_idx = {ch: i+2 for i, ch in enumerate(all_chars)}
char_to_idx["<PAD>"] = 0
char_to_idx["<UNK>"] = 1

In [35]:
imdbdataset = IMDBDataset(imdb_data_path, char_to_idx)
counter = 0
for sample, label in imdbdataset:
    print(sample.shape)
    print(label)
    counter +=1

    if counter ==3:
        break

torch.Size([1000])
tensor(1)
torch.Size([1000])
tensor(1)
torch.Size([1000])
tensor(1)


In [36]:
# Samples
sample1 = torch.tensor([1, 2, 3])
sample2 = torch.tensor([4, 5])
sample3 = torch.tensor([6])

In [37]:
batch = torch.stack([sample1, sample2, sample3])  # ❌ will raise error

RuntimeError: stack expects each tensor to be equal size, but got [3] at entry 0 and [2] at entry 1

In [38]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    # batch = list of samples = [(input1, label1), (input2, label2), ...]
    sequences, labels = zip(*batch) # unzip batch
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    labels = torch.tensor(labels)
    return padded_sequences, labels

In [40]:
dataloader = DataLoader(imdbdataset, batch_size=4, collate_fn=collate_fn)